# TrafficFlowBench 2026 – full pipeline
Task 1 (LightGBM reconstruction) · Task 2 (queue forecast, IoU-tuned) · Task 3 (follows Task 1) · Task 4 (official baseline for now)

**Requires:** competition data attached + Internet ON (to clone the official repo).

Run order: cells top to bottom. Use the *Quick test* cell first, then the *Full run* cell.

In [ ]:
import os, sys, re, time, argparse, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

RELEASE = Path(os.environ.get(
    "TFB_RELEASE",
    "/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public"))
WORK = Path(os.environ.get("TFB_WORK", "/kaggle/working"))
REPO = WORK / "trafficflowbench-public"
OUT = WORK / "tfb_out"
SLOTS = 288
HORIZ = 6
HIST = 12
SEED = 42
REGIME_CODE = {"R1": 1, "R2": 2, "R3": 3}

CFG = dict(
    t1_sample_frac=0.10,      # fraction of target cells sampled per train day
    t1_holdout_days=30,
    t1_rounds=1500,
    t2_stride=6,              # origin every 30 min when generating training windows
    t2_holdout_days=45,
    t2_max_rows_per_panel=350_000,
    t2_max_holdout_windows=400,
    t2_rounds=800,
)


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)

## helpers: repo, generic numerics

In [ ]:
def ensure_repo():
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jacky850/trafficflowbench-public.git",
                        str(REPO)], check=True)
    for p in (REPO / "src", REPO / "src" / "task1"):
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))


def nanmean_ax(a, axis):
    fin = np.isfinite(a)
    s = np.where(fin, a, 0).sum(axis)
    c = fin.sum(axis)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)


def prev_next(M, maxgap):
    """Nearest visible value strictly before / after along axis 0 (+ gap in steps)."""
    T, U = M.shape
    ar = np.arange(T)[:, None]
    vis = np.isfinite(M)
    BIG = 10 ** 6
    last = np.maximum.accumulate(np.where(vis, ar, -1), axis=0)
    lastp = np.empty_like(last); lastp[0] = -1; lastp[1:] = last[:-1]
    nxt = np.minimum.accumulate(np.where(vis, ar, BIG)[::-1], axis=0)[::-1]
    nxtn = np.empty_like(nxt); nxtn[-1] = BIG; nxtn[:-1] = nxt[1:]
    cols = np.arange(U)[None, :]
    gp = (ar - lastp).astype(np.float32)
    gn = (nxtn - ar).astype(np.float32)
    okp = (lastp >= 0) & (gp <= maxgap)
    okn = (nxtn < BIG) & (gn <= maxgap)
    vp = np.where(okp, M[np.clip(lastp, 0, T - 1), cols], np.nan).astype(np.float32)
    vn = np.where(okn, M[np.clip(nxtn, 0, T - 1), cols], np.nan).astype(np.float32)
    return vp, np.where(okp, gp, np.nan), vn, np.where(okn, gn, np.nan)


def interp(vp, gp, vn, gn):
    both = np.isfinite(vp) & np.isfinite(vn)
    w = np.where(both, gn / np.where(both, gp + gn, 1), np.nan)
    return np.where(both, vp * w + vn * (1 - w),
                    np.where(np.isfinite(vp), vp, vn)).astype(np.float32)


def loo_profile(X, wd, k):
    """X [nd,288,K] with NaN. Weekday x slot mean, smoothed +-k slots.
    Returns leave-one-day-out profile [nd,288,K] and the full profile [7,288,K]."""
    nd, T, K = X.shape
    fin = np.isfinite(X)
    v = np.where(fin, X, 0).astype(np.float64)
    c = fin.astype(np.float64)

    def sm(a):
        if k == 0:
            return a
        out = a.copy()
        for s in range(1, k + 1):
            out += np.roll(a, s, axis=1) + np.roll(a, -s, axis=1)
        return out
    sv, sc = sm(v), sm(c)
    tv = np.zeros((7, T, K)); tc = np.zeros((7, T, K))
    for w in range(7):
        m = wd == w
        if m.any():
            tv[w] = sv[m].sum(0); tc[w] = sc[m].sum(0)
    lv = tv[wd] - sv; lc = tc[wd] - sc
    loo = np.where(lc > 0, lv / np.maximum(lc, 1), np.nan).astype(np.float32)
    full = np.where(tc > 0, tv / np.maximum(tc, 1), np.nan).astype(np.float32)
    return loo, full


def day_of(path):
    m = re.search(r"(\d{4})_(\d{2})_(\d{2})", Path(path).stem)
    return pd.Timestamp(f"{m[1]}-{m[2]}-{m[3]}", tz="UTC")


def parquet_cols(path, wanted):
    import pyarrow.parquet as pq
    have = set(pq.read_schema(path).names)
    return [c for c in wanted if c in have]

## Panel container

In [ ]:
class Panel:
    def __init__(self, name, pid):
        import score_task1
        self.score = score_task1
        self.name, self.pid = name, pid
        self.dir = RELEASE / "corridors" / name
        self.links = pd.read_csv(self.dir / "network" / "links.csv")
        self.links["link_id"] = self.links["link_id"].astype(str)
        self.link_ids = self.links["link_id"].tolist()          # assumed upstream -> downstream
        self.link_idx = {l: i for i, l in enumerate(self.link_ids)}
        self.L = len(self.link_ids)
        self.vf_link = pd.to_numeric(self.links["free_speed_kmh"], errors="coerce").to_numpy(float)
        self.cap_link = pd.to_numeric(self.links["capacity_vph"], errors="coerce").to_numpy(float)
        self._thr()
        self.lanes_by_station = score_task1.station_lanes(self.dir)
        self.load_truth()

    def _thr(self):
        thr = 0.6 * self.vf_link
        try:
            from task2.queue_utils import thresholds
            tmap, _ = thresholds(self.dir, self.links)
            tmap = {str(k): float(v) for k, v in tmap.items()}
            thr = np.array([tmap.get(l, 0.6 * self.vf_link[i]) for i, l in enumerate(self.link_ids)])
        except Exception as e:
            log(f"  [{self.name}] thresholds() fallback 0.6*vf ({e})")
        self.thr = thr

    def load_truth(self):
        files = sorted((self.dir / "train" / "mainline_states").glob("**/*.parquet"))
        wanted = ["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph",
                  "occupancy", "pct_observed"]
        cols = parquet_cols(files[0], wanted)
        first = pd.read_parquet(files[0], columns=["station_id", "link_id"]).astype(str)
        pairs = first.drop_duplicates().copy()
        pairs["rk"] = pairs["link_id"].map(self.link_idx).fillna(10 ** 6)
        pairs = pairs.sort_values(["rk", "station_id"]).reset_index(drop=True)
        self.u_station = pairs["station_id"].tolist()
        self.u_link = pairs["link_id"].tolist()
        self.U = len(pairs)
        self.uidx = {f"{s}|{l}": i for i, (s, l) in enumerate(zip(self.u_station, self.u_link))}
        self.u_lidx = np.array([self.link_idx.get(l, -1) for l in self.u_link])
        self.u_vf = np.array([self.vf_link[i] if i >= 0 else 100. for i in self.u_lidx])
        self.u_cap = np.array([self.cap_link[i] if i >= 0 else 2000. for i in self.u_lidx])
        self.u_pos = np.where(self.u_lidx >= 0, self.u_lidx / max(self.L - 1, 1), 0.5)
        self.u_lanes = self.lane_vec(pd.Series(self.u_station), pd.Series(self.u_link))
        dates = [day_of(f) for f in files]
        self.d0 = min(dates)
        self.nd = (max(dates) - self.d0).days + 1
        T = self.nd * SLOTS
        S = np.full((T, self.U), np.nan, np.float32)
        F = S.copy(); O = S.copy()
        for f in files:
            df = pd.read_parquet(f, columns=cols)
            ts = pd.to_datetime(df["timestamp"], utc=True)
            d = (ts.dt.floor("D") - self.d0).dt.days.to_numpy()
            row = d * SLOTS + (ts.dt.hour * 12 + ts.dt.minute // 5).to_numpy()
            key = df["station_id"].astype(str) + "|" + df["link_id"].astype(str)
            u = key.map(self.uidx).to_numpy(dtype=float)
            ok = np.isfinite(u)
            sp = pd.to_numeric(df["speed_kmh"], errors="coerce").to_numpy(float)
            fl = pd.to_numeric(df["flow_vph"], errors="coerce").to_numpy(float)
            oc = (pd.to_numeric(df["occupancy"], errors="coerce").to_numpy(float)
                  if "occupancy" in df else np.full(len(df), np.nan))
            pc = (pd.to_numeric(df["pct_observed"], errors="coerce").to_numpy(float)
                  if "pct_observed" in df else np.full(len(df), 100.))
            good = ok & (pc >= 75) & np.isfinite(sp) & np.isfinite(fl)
            r, c = row[good], u[good].astype(int)
            S[r, c] = sp[good]; F[r, c] = fl[good]; O[r, c] = oc[good]
        self.S, self.F, self.O = S, F, O
        self.Fl = F / self.u_lanes[None, :]                      # flow per lane
        self.wd_day = ((self.d0.weekday() + np.arange(self.nd)) % 7).astype(int)
        log(f"  [{self.name}] truth loaded: {self.nd} days, {self.U} units, {self.L} links")

    def lane_vec(self, st, li):
        v = self.score.lane_vector(st.reset_index(drop=True), li.reset_index(drop=True),
                                   self.lanes_by_station)
        v = np.asarray(v, dtype=float)
        v[~np.isfinite(v) | (v <= 0)] = 3.0
        return v

    # ---- Task 1 profiles -------------------------------------------------
    def build_t1_profiles(self):
        nd, U = self.nd, self.U
        X = np.stack([self.S.reshape(nd, SLOTS, U), self.Fl.reshape(nd, SLOTS, U)], axis=-1)
        X = X.reshape(nd, SLOTS, U * 2)
        self.prof = {}
        for k in (0, 2):
            loo, full = loo_profile(X, self.wd_day, k)
            self.prof[k] = (loo.reshape(nd, SLOTS, U, 2), full.reshape(7, SLOTS, U, 2))
        self.unit_mean = np.stack([nanmean_ax(self.S, 0), nanmean_ax(self.Fl, 0)], -1)  # [U,2]

    def hist_t1(self, k, day_idx=None, weekday=None):
        if day_idx is not None:
            h = self.prof[k][0][day_idx]
        else:
            h = self.prof[k][1][weekday]
        h = np.where(np.isfinite(h), h, self.unit_mean[None, :, :])
        return h[..., 0], h[..., 1]

    # ---- Task 2 link-level matrices ----------------------------------------
    def build_link_matrices(self):
        T = self.nd * SLOTS
        def agg(M):
            out = np.full((T, self.L), np.nan, np.float32)
            for l in range(self.L):
                cols = np.where(self.u_lidx == l)[0]
                if len(cols):
                    out[:, l] = nanmean_ax(M[:, cols], 1)
            return out
        self.Sl, self.Ol = agg(self.S), agg(self.O)
        self.Fl_link = agg(self.F) / self.cap_link[None, :]
        self.Rl = self.Sl / self.thr[None, :]
        self.Ql = np.where(np.isfinite(self.Rl), (self.Rl <= 1.0).astype(np.float32), np.nan)
        nd, L = self.nd, self.L
        Rc = np.clip(self.Rl, 0, 3).reshape(nd, SLOTS, L)
        Qd = self.Ql.reshape(nd, SLOTS, L)
        self.hq_loo, self.hq_full = loo_profile(Qd, self.wd_day, 2)
        self.hr_loo, self.hr_full = loo_profile(Rc, self.wd_day, 2)
        self.hq_loo = self.hq_loo.reshape(-1, L); self.hr_loo = self.hr_loo.reshape(-1, L)

## TASK 1

In [ ]:
def to_matrix(df_key_u, slot, vals, U):
    M = np.full((SLOTS, U), np.nan, np.float32)
    ok = np.isfinite(df_key_u)
    M[slot[ok], df_key_u[ok].astype(int)] = vals[ok]
    return M


def load_masked_day(P, path):
    wanted = ["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph", "occupancy"]
    df = pd.read_parquet(path, columns=parquet_cols(path, wanted))
    ts = pd.to_datetime(df["timestamp"], utc=True)
    slot = (ts.dt.hour * 12 + ts.dt.minute // 5).to_numpy()
    key = df["station_id"].astype(str) + "|" + df["link_id"].astype(str)
    u = key.map(P.uidx).to_numpy(dtype=float)
    sp = pd.to_numeric(df["speed_kmh"], errors="coerce").to_numpy(float)
    fl = pd.to_numeric(df["flow_vph"], errors="coerce").to_numpy(float)
    oc = (pd.to_numeric(df["occupancy"], errors="coerce").to_numpy(float)
          if "occupancy" in df else np.full(len(df), np.nan))
    both = np.isfinite(sp) & np.isfinite(fl)
    sp = np.where(both, sp, np.nan); fl = np.where(both, fl, np.nan)
    Vs = to_matrix(u, slot, sp, P.U)
    Vf = to_matrix(u, slot, fl, P.U) / P.u_lanes[None, :]
    Vo = to_matrix(u, slot, oc, P.U)
    return Vs, Vf.astype(np.float32), Vo


def spatial_pn(M, maxgap):
    a, b, c, d = prev_next(M.T, maxgap)
    return a.T, b.T, c.T, d.T


def t1_maps(Vs, Vf, Vo, hs, hf, hs0, hf0):
    maps = {}
    for nm, V, h, h0 in (("s", Vs, hs, hs0), ("f", Vf, hf, hf0)):
        maps[f"{nm}_hist"] = h
        maps[f"{nm}_hist0"] = h0
        vp, gp, vn, gn = prev_next(V, 12)
        maps[f"{nm}_tprev"], maps[f"{nm}_tgp"], maps[f"{nm}_tnext"], maps[f"{nm}_tgn"] = vp, gp, vn, gn
        maps[f"{nm}_tint"] = interp(vp, gp, vn, gn)
        R = V - h
        rp, rgp, rn, rgn = prev_next(R, 12)
        maps[f"{nm}_tres"] = interp(rp, rgp, rn, rgn)
        maps[f"{nm}_hist_tres"] = h + maps[f"{nm}_tres"]
        sp_, sgp, sn_, sgn = spatial_pn(V, 3)
        maps[f"{nm}_sprev"], maps[f"{nm}_sgp"], maps[f"{nm}_snext"], maps[f"{nm}_sgn"] = sp_, sgp, sn_, sgn
        maps[f"{nm}_sint"] = interp(sp_, sgp, sn_, sgn)
        Rp, Rgp, Rn, Rgn = spatial_pn(R, 3)
        maps[f"{nm}_sres"] = interp(Rp, Rgp, Rn, Rgn)
        maps[f"{nm}_hist_sres"] = h + maps[f"{nm}_sres"]
        cs = nanmean_ax(R, 1)                                   # corridor residual at t
        kern = np.ones(13) / 13
        csf = np.where(np.isfinite(cs), cs, 0); csc = np.isfinite(cs).astype(float)
        cs13 = np.convolve(csf, kern, "same") / np.maximum(np.convolve(csc, kern, "same"), 1e-6)
        maps[f"{nm}_cs"] = np.broadcast_to(cs[:, None], Vs.shape)
        maps[f"{nm}_cs13"] = np.broadcast_to(cs13[:, None], Vs.shape)
    maps["o_self"] = Vo
    op, ogp, on_, ogn = prev_next(Vo, 12)
    maps["o_tint"] = interp(op, ogp, on_, ogn)
    maps["o_sint"] = interp(*spatial_pn(Vo, 3))
    return maps


def t1_rows(P, maps, ti, ui, regime):
    cols, names = [], []
    for k, m in maps.items():
        cols.append(np.asarray(m)[ti, ui].astype(np.float32)); names.append(k)
    n = len(ti)
    stat = {"slot": ti, "u_vf": P.u_vf[ui], "u_lanes": P.u_lanes[ui], "u_cap": P.u_cap[ui],
            "u_pos": P.u_pos[ui], "regime": np.full(n, REGIME_CODE.get(regime, 0)),
            "panel_id": np.full(n, P.pid)}
    for k, v in stat.items():
        cols.append(np.asarray(v, np.float32)); names.append(k)
    return np.stack(cols, 1), names


def read_template(path):
    t = pd.read_csv(path)
    t["ts"] = pd.to_datetime(t["timestamp"], utc=True)
    t["date"] = t["ts"].dt.floor("D")
    t["slot"] = (t["ts"].dt.hour * 12 + t["ts"].dt.minute // 5).astype(int)
    return t


def masked_index(P, split, regime):
    root = P.dir / split / "mainline_states_masked" / f"mask_regime={regime}"
    return {day_of(f): f for f in root.glob("**/*.parquet")}


def t1_collect(P, split, rng, sample_frac=None, holdout_days=0):
    """Returns feature rows. For split=='train' also truth targets & holdout flag."""
    tpl = read_template(RELEASE / "task1" / P.name / split / "sample_submission_state.csv")
    tpl["u"] = (tpl["station_id"].astype(str) + "|" + tpl["link_id"].astype(str)).map(P.uidx)
    idx_cache = {}
    Xs, ys, meta, rows = [], [], [], []
    names = None
    for (reg, date), g in tpl.groupby(["mask_regime", "date"], sort=False):
        if reg not in idx_cache:
            idx_cache[reg] = masked_index(P, split, reg)
        path = idx_cache[reg].get(date)
        if path is None:
            continue
        gg = g[g["u"].notna()]
        if sample_frac is not None:
            gg = gg.sample(frac=sample_frac, random_state=int(rng.integers(1 << 30)))
        if gg.empty:
            continue
        Vs, Vf, Vo = load_masked_day(P, path)
        if split == "train":
            d = (date - P.d0).days
            hs, hf = P.hist_t1(2, day_idx=d); hs0, hf0 = P.hist_t1(0, day_idx=d)
        else:
            hs, hf = P.hist_t1(2, weekday=date.weekday()); hs0, hf0 = P.hist_t1(0, weekday=date.weekday())
        maps = t1_maps(Vs, Vf, Vo, hs, hf, hs0, hf0)
        ti = gg["slot"].to_numpy(); ui = gg["u"].to_numpy().astype(int)
        X, names = t1_rows(P, maps, ti, ui, reg)
        Xs.append(X)
        rows.append(gg)
        if split == "train":
            r = d * SLOTS + ti
            ys.append(np.stack([P.S[r, ui], P.Fl[r, ui]], 1))
            meta.append(np.stack([np.full(len(ti), d), np.full(len(ti), REGIME_CODE[reg])], 1))
    X = np.concatenate(Xs) if Xs else np.zeros((0, 1), np.float32)
    if split == "train":
        return X, np.concatenate(ys), np.concatenate(meta), names
    return X, names, (pd.concat(rows) if rows else tpl.iloc[:0])


T1_PARAMS = dict(objective="regression", learning_rate=0.06, num_leaves=127,
                 min_data_in_leaf=80, feature_fraction=0.8, bagging_fraction=0.8,
                 bagging_freq=1, lambda_l2=2.0, verbose=-1, num_threads=os.cpu_count() or 4,
                 seed=SEED)


def train_task1(panels, quick=False):
    import lightgbm as lgb
    rng = np.random.default_rng(SEED)
    Xs, ys, ms = [], [], []
    names = None
    for P in panels:
        P.build_t1_profiles()
        X, y, m, names = t1_collect(P, "train", rng, CFG["t1_sample_frac"] * (0.4 if quick else 1))
        ok = np.isfinite(y).all(1)
        Xs.append(X[ok]); ys.append(y[ok]); ms.append(m[ok])
        log(f"  T1 [{P.name}] train rows {ok.sum():,}")
    X = np.concatenate(Xs); y = np.concatenate(ys); m = np.concatenate(ms)
    last_day = np.concatenate([np.full(len(a), P.nd) for a, P in zip(Xs, panels)])
    hold = m[:, 0] >= (last_day - CFG["t1_holdout_days"])
    cat = [names.index("panel_id")]
    models = {}
    for j, tgt in enumerate(("speed", "flow_per_lane")):
        dtr = lgb.Dataset(X[~hold], y[~hold, j], categorical_feature=cat, free_raw_data=False)
        dva = lgb.Dataset(X[hold], y[hold, j], reference=dtr)
        b = lgb.train(T1_PARAMS, dtr, CFG["t1_rounds"] if not quick else 200, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(60), lgb.log_evaluation(100)])
        best = max(b.best_iteration, 50)
        pr = b.predict(X[hold], num_iteration=best)
        rm = float(np.sqrt(np.mean((pr - y[hold, j]) ** 2)))
        log(f"  T1 {tgt}: holdout RMSE {rm:.3f} (best_iter {best})")
        # refit on everything
        full = lgb.Dataset(X, y[:, j], categorical_feature=cat)
        models[tgt] = lgb.train(T1_PARAMS, full, int(best * 1.1))
    return models, names


def predict_task1(panels, models, split):
    rng = np.random.default_rng(SEED)
    parts = []
    for P in panels:
        tpl = read_template(RELEASE / "task1" / P.name / split / "sample_submission_state.csv")
        tpl["u"] = (tpl["station_id"].astype(str) + "|" + tpl["link_id"].astype(str)).map(P.uidx)
        X, names, order = t1_collect(P, split, rng)
        assert len(order) == len(X)
        sp = np.clip(models["speed"].predict(X), 0, 1.15 * P.u_vf[order["u"].to_numpy().astype(int)])
        fl = np.clip(models["flow_per_lane"].predict(X), 0, None)
        res = order[["panel", "timestamp", "station_id", "link_id", "mask_regime"]].copy()
        res["speed_kmh"] = sp
        res["flow_vph"] = fl * P.u_lanes[order["u"].to_numpy().astype(int)]
        # rows without a unit / masked file -> fallback to unit means
        missing = tpl.loc[~tpl.index.isin(order.index), ["panel", "timestamp", "station_id", "link_id", "mask_regime"]].copy()
        if len(missing):
            lanes = P.lane_vec(missing["station_id"].astype(str), missing["link_id"].astype(str))
            missing["speed_kmh"] = float(np.nanmean(P.S))
            missing["flow_vph"] = float(np.nanmean(P.Fl)) * lanes
            res = pd.concat([res, missing])
        res["split"] = split
        parts.append(res)
        log(f"  T1 predict [{P.name}/{split}] rows {len(res):,} (fallback {len(missing):,})")
    out = pd.concat(parts, ignore_index=True)
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True)
    out["station_id"] = out["station_id"].astype(str); out["link_id"] = out["link_id"].astype(str)
    return out

## TASK 2

In [ ]:
def shiftL(x, k):
    out = np.full_like(x, np.nan)
    if k > 0:
        out[:, :-k] = x[:, k:]
    elif k < 0:
        out[:, -k:] = x[:, :k]
    else:
        out = x.copy()
    return out


def last_valid(a):
    fin = np.isfinite(a)
    idx = np.where(fin, np.arange(a.shape[1])[None, :, None], -1).max(axis=1)
    val = np.take_along_axis(a, np.clip(idx, 0, None)[:, None, :], axis=1)[:, 0, :]
    return np.where(idx >= 0, val, np.nan), np.where(idx >= 0, a.shape[1] - 1 - idx, np.nan)


T2_NAMES = None


def t2_features(P, rh, oh, fh, hq_o, hq_t, hr_t, cond, slot_o, wd_o):
    """rh,oh,fh [n,13,L]; hq_o [n,L]; hq_t,hr_t [n,6,L]; cond,slot_o,wd_o [n].
    returns X [n,L,6,nf]"""
    global T2_NAMES
    n, _, L = rh.shape
    f = {}
    r_last, age = last_valid(rh)
    f["r_last"], f["age"] = r_last, age
    f["r_T"], f["r_m1"] = rh[:, 12], rh[:, 11]
    f["r_m3"] = nanmean_ax(rh[:, 10:13], 1)
    f["r_min"] = np.where(np.isfinite(rh).any(1), np.nanmin(np.where(np.isfinite(rh), rh, np.inf), 1), np.nan)
    f["r_min"] = np.where(np.isfinite(f["r_min"]) & (f["r_min"] < 1e5), f["r_min"], np.nan)
    f["r_mean"] = nanmean_ax(rh, 1)
    f["sl1"] = r_last - nanmean_ax(rh[:, 6:9], 1)
    f["sl2"] = r_last - nanmean_ax(rh[:, 0:3], 1)
    f["nq_hist"] = (rh <= 1).sum(1).astype(np.float32)
    f["q_last"] = np.where(np.isfinite(r_last), (r_last <= 1).astype(np.float32), np.nan)
    o_last, _ = last_valid(oh); f_last, _ = last_valid(fh)
    f["o_last"], f["f_last"], f["o_m3"] = o_last, f_last, nanmean_ax(oh[:, 10:13], 1)
    for k in (1, 2, 3):
        f[f"ds{k}"] = shiftL(r_last, k)
    for k in (1, 2):
        f[f"us{k}"] = shiftL(r_last, -k)
    f["dsmin"] = np.nanmin(np.stack([f["ds1"], f["ds2"], f["ds3"]]), 0) if False else \
        np.where(np.isfinite(np.stack([f["ds1"], f["ds2"], f["ds3"]])).any(0),
                 np.nanmin(np.where(np.isfinite(np.stack([f["ds1"], f["ds2"], f["ds3"]])),
                                    np.stack([f["ds1"], f["ds2"], f["ds3"]]), np.inf), 0), np.nan)
    f["dsmin"] = np.where(f["dsmin"] < 1e5, f["dsmin"], np.nan)
    cmin = np.where(np.isfinite(r_last).any(1), np.nanmin(np.where(np.isfinite(r_last), r_last, np.inf), 1), np.nan)
    f["c_min"] = np.broadcast_to(cmin[:, None], (n, L))
    f["c_mean"] = np.broadcast_to(nanmean_ax(r_last, 1)[:, None], (n, L))
    f["c_nq"] = np.broadcast_to((r_last <= 1).sum(1)[:, None], (n, L))
    f["c_obs"] = np.broadcast_to(np.isfinite(rh).mean((1, 2))[:, None], (n, L))
    f["cond"] = np.broadcast_to(cond[:, None], (n, L))
    f["slot_o"] = np.broadcast_to(slot_o[:, None], (n, L))
    f["wd_o"] = np.broadcast_to(wd_o[:, None], (n, L))
    f["hq_o"] = hq_o
    f["vf"] = np.broadcast_to(P.vf_link[None, :], (n, L))
    f["thr"] = np.broadcast_to(P.thr[None, :], (n, L))
    f["cap"] = np.broadcast_to(P.cap_link[None, :], (n, L))
    f["pos"] = np.broadcast_to((np.arange(L) / max(L - 1, 1))[None, :], (n, L))
    f["pid"] = np.full((n, L), P.pid)
    base = np.stack([np.asarray(v, np.float32) for v in f.values()], -1)       # [n,L,nb]
    base = np.repeat(base[:, :, None, :], HORIZ, axis=2)                        # [n,L,6,nb]
    h = np.broadcast_to(np.arange(1, HORIZ + 1)[None, None, :], (n, L, HORIZ))
    hq = np.transpose(hq_t, (0, 2, 1)); hr = np.transpose(hr_t, (0, 2, 1))      # [n,L,6]
    hq_ds1 = np.stack([shiftL(hq_t[:, i], 1) for i in range(HORIZ)], 2)
    extra = np.stack([h, hq, hr, hq_ds1], -1).astype(np.float32)
    X = np.concatenate([base, extra], -1)
    T2_NAMES = list(f.keys()) + ["h", "hq_t", "hr_t", "hq_t_ds1"]
    return X


def t2_train_windows(P):
    Ql, Rl = P.Ql, P.Rl
    T = Ql.shape[0]
    I = np.arange(HIST, T - HORIZ, CFG["t2_stride"])
    hist_idx = I[:, None] + np.arange(-HIST, 1)[None, :]
    fut_idx = I[:, None] + np.arange(1, HORIZ + 1)[None, :]
    Qh, Qf = Ql[hist_idx], Ql[fut_idx]
    anyf = (Qf == 1).any((1, 2))
    hist_any = (Qh[:, :HIST] == 1).any((1, 2))
    at_T = (Qh[:, HIST] == 1).any(1)
    obs = np.isfinite(Rl[hist_idx]).mean((1, 2))
    cond = np.where(at_T, 1, 0)
    keep = anyf & ((~hist_any) | at_T) & (obs > 0.3)
    return I[keep], hist_idx[keep], fut_idx[keep], cond[keep]


def t2_build(P, I, hist_idx, fut_idx, cond):
    X = t2_features(P, P.Rl[hist_idx], P.Ol[hist_idx], P.Fl_link[hist_idx],
                    P.hq_loo[I], P.hq_loo[fut_idx], P.hr_loo[fut_idx],
                    cond.astype(np.float32), (I % SLOTS).astype(np.float32),
                    P.wd_day[np.clip(I // SLOTS, 0, P.nd - 1)].astype(np.float32))
    y = np.transpose(P.Ql[fut_idx], (0, 2, 1))                     # [n,L,6]
    return X, y


T2_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=100,
                 feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0,
                 verbose=-1, num_threads=os.cpu_count() or 4, seed=SEED)


def iou_grid(prob, y, cond):
    """tune per-condition threshold and min-cells. prob,y [n,L,6]"""
    known = np.isfinite(y)
    yt = (y == 1)
    n = len(prob)
    pm = np.where(known, prob, -1).reshape(n, -1)
    rank = np.argsort(np.argsort(-pm, axis=1), axis=1).reshape(prob.shape)
    best = {}
    for c in (0, 1):
        m = cond == c
        if m.sum() == 0:
            best[c] = (0.3, 3, 0.0); continue
        bs = (-1, None)
        for t in np.arange(0.05, 0.65, 0.025):
            for kmin in (1, 2, 3, 5, 8, 12, 20):
                pred = (prob[m] >= t) & known[m]
                empty = pred.sum((1, 2)) == 0
                pred = pred | (empty[:, None, None] & (rank[m] < kmin))
                inter = (pred & yt[m]).sum((1, 2)); uni = (pred | yt[m]).sum((1, 2))
                iou = np.where(uni > 0, inter / np.maximum(uni, 1), 1.0).mean()
                if iou > bs[0]:
                    bs = (iou, (float(t), kmin))
        best[c] = (bs[1][0], bs[1][1], float(bs[0]))
        log(f"  T2 cond={c}: best thr {bs[1][0]:.3f} kmin {bs[1][1]} holdout IoU {bs[0]:.4f} (n={m.sum()})")
    return best


def train_task2(panels, quick=False):
    import lightgbm as lgb
    rng = np.random.default_rng(SEED)
    Xtr, ytr, Xho, yho, cho = [], [], [], [], []
    for P in panels:
        if P.name.startswith("D12_I405"):
            continue
        P.build_link_matrices()
        I, hi, fi, cond = t2_train_windows(P)
        day = I // SLOTS
        is_ho = day >= (P.nd - CFG["t2_holdout_days"])
        log(f"  T2 [{P.name}] windows {len(I):,} (onset {(cond == 0).sum()}, ongoing {(cond == 1).sum()}), holdout {is_ho.sum()}")
        tr_idx = np.where(~is_ho)[0]
        budget = CFG["t2_max_rows_per_panel"] * (0.3 if quick else 1)
        keep_p = min(1.0, budget / max(len(tr_idx) * P.L * HORIZ, 1))
        for s in range(0, len(tr_idx), 400):
            sel = tr_idx[s:s + 400]
            X, y = t2_build(P, I[sel], hi[sel], fi[sel], cond[sel])
            X = X.reshape(-1, X.shape[-1]); y = y.reshape(-1)
            m = np.isfinite(y) & (rng.random(len(y)) < keep_p)
            Xtr.append(X[m]); ytr.append(y[m])
        ho_idx = np.where(is_ho)[0]
        if len(ho_idx) > CFG["t2_max_holdout_windows"]:
            ho_idx = rng.choice(ho_idx, CFG["t2_max_holdout_windows"], replace=False)
        if len(ho_idx):
            X, y = t2_build(P, I[ho_idx], hi[ho_idx], fi[ho_idx], cond[ho_idx])
            Xho.append(X); yho.append(y); cho.append(cond[ho_idx])
    Xtr = np.concatenate(Xtr); ytr = np.concatenate(ytr)
    log(f"  T2 train rows {len(Xtr):,}, positive rate {ytr.mean():.4f}")
    cat = [T2_NAMES.index("pid")]
    hx = np.concatenate([x.reshape(-1, x.shape[-1]) for x in Xho])
    hy = np.concatenate([y.reshape(-1) for y in yho])
    mk = np.isfinite(hy)
    dtr = lgb.Dataset(Xtr, ytr, categorical_feature=cat, free_raw_data=False)
    dva = lgb.Dataset(hx[mk], hy[mk], reference=dtr)
    b = lgb.train(T2_PARAMS, dtr, CFG["t2_rounds"] if not quick else 150, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])
    best = max(b.best_iteration, 50)
    # tune decision rule on holdout windows
    probs = [b.predict(x.reshape(-1, x.shape[-1]), num_iteration=best).reshape(x.shape[:-1]) for x in Xho]
    Lmax = max(p.shape[1] for p in probs)          # panels have different numbers of links -> pad
    def _pad(a, v):
        return np.pad(a, ((0, 0), (0, Lmax - a.shape[1]), (0, 0)), constant_values=v)
    prob = np.concatenate([_pad(p, 0.0) for p in probs])
    yh = np.concatenate([_pad(y, np.nan) for y in yho]); ch = np.concatenate(cho)
    rule = iou_grid(prob, yh, ch)
    log(f"  T2 holdout mean IoU (cond-avg) ~ {np.mean([rule[0][2], rule[1][2]]):.4f}")
    full = lgb.Dataset(Xtr, ytr, categorical_feature=cat)
    model = lgb.train(T2_PARAMS, full, int(best * 1.1))
    return model, rule


def predict_task2(panels, model, rule, split):
    parts = []
    for P in panels:
        d = RELEASE / "task2" / P.name / split
        if not d.exists():
            continue
        win = pd.read_csv(d / "window_index.csv")
        hist = pd.read_parquet(d / "window_history.parquet")
        tpl = pd.read_csv(d / "sample_submission_queue.csv")
        win["window_id"] = win["window_id"].astype(str)
        hist["window_id"] = hist["window_id"].astype(str); tpl["window_id"] = tpl["window_id"].astype(str)
        win["origin"] = pd.to_datetime(win["forecast_origin"], utc=True)
        wmap = {w: i for i, w in enumerate(win["window_id"])}
        n = len(win)
        hist["ts"] = pd.to_datetime(hist["timestamp"], utc=True)
        hist["w"] = hist["window_id"].map(wmap)
        oidx = pd.DatetimeIndex(win["origin"])

        def _steps(ts, w):
            out = pd.Series(np.nan, index=ts.index)
            ok = w.notna()
            if ok.any():
                o = pd.Series(oidx[w[ok].astype(int).to_numpy()], index=ts.index[ok])
                out[ok] = ((ts[ok] - o) / pd.Timedelta(minutes=5)).round()
            return out
        hist["k"] = _steps(hist["ts"], hist["w"]) + HIST
        hist["l"] = hist["link_id"].astype(str).map(P.link_idx)
        elig = hist["is_score_eligible"].astype(bool) if "is_score_eligible" in hist else True
        hist = hist[(hist["k"] >= 0) & (hist["k"] <= HIST) & hist["l"].notna() & hist["w"].notna()]
        rh = np.full((n, HIST + 1, P.L), np.nan, np.float32); oh = rh.copy(); fh = rh.copy()
        w_, k_, l_ = hist["w"].to_numpy(int), hist["k"].to_numpy(int), hist["l"].to_numpy(int)
        el = elig.loc[hist.index].to_numpy() if hasattr(elig, "loc") else np.ones(len(hist), bool)
        sp = pd.to_numeric(hist["speed_kmh"], errors="coerce").to_numpy(float)
        sp = np.where(el, sp, np.nan)
        rh[w_, k_, l_] = sp / P.thr[l_]
        if "occupancy" in hist:
            oh[w_, k_, l_] = np.where(el, pd.to_numeric(hist["occupancy"], errors="coerce").to_numpy(float), np.nan)
        if "flow_vph" in hist:
            fh[w_, k_, l_] = np.where(el, pd.to_numeric(hist["flow_vph"], errors="coerce").to_numpy(float), np.nan) / P.cap_link[l_]
        oi = pd.DatetimeIndex(win["origin"])
        wd_o = np.asarray(oi.weekday); slot_o = np.asarray(oi.hour * 12 + oi.minute // 5)
        wd_t = np.stack([np.asarray((oi + pd.Timedelta(minutes=5 * h)).weekday) for h in range(1, HORIZ + 1)], 1)
        slot_t = np.stack([np.asarray(((oi + pd.Timedelta(minutes=5 * h)).hour * 12 + (oi + pd.Timedelta(minutes=5 * h)).minute // 5)) for h in range(1, HORIZ + 1)], 1)
        hq_o = P.hq_full[wd_o, slot_o]
        hq_t = P.hq_full[wd_t, slot_t]; hr_t = P.hr_full[wd_t, slot_t]
        cond = (win["condition"].astype(str).str.contains("ongoing")).to_numpy().astype(np.float32)
        X = t2_features(P, rh, oh, fh, hq_o, hq_t, hr_t, cond, slot_o.astype(np.float32), wd_o.astype(np.float32))
        prob = model.predict(X.reshape(-1, X.shape[-1])).reshape(X.shape[:-1])        # [n,L,6]
        pm = prob.reshape(n, -1)
        rank = np.argsort(np.argsort(-pm, axis=1), axis=1).reshape(prob.shape)
        pred = np.zeros(prob.shape, bool)
        for i in range(n):
            t, kmin, _ = rule[int(cond[i])]
            p = prob[i] >= t
            if not p.any():
                p = rank[i] < kmin
            pred[i] = p
        tpl["ts"] = pd.to_datetime(tpl["timestamp"], utc=True)
        tpl["w"] = tpl["window_id"].map(wmap)
        tpl["l"] = tpl["link_id"].astype(str).map(P.link_idx)
        tpl["h"] = _steps(tpl["ts"], tpl["w"]) - 1
        ok = tpl["w"].notna() & tpl["l"].notna() & (tpl["h"] >= 0) & (tpl["h"] < HORIZ)
        qp = np.zeros(len(tpl), int)
        qp[ok.to_numpy()] = pred[tpl.loc[ok, "w"].astype(int), tpl.loc[ok, "l"].astype(int), tpl.loc[ok, "h"].astype(int)]
        res = pd.DataFrame({"split": split, "window_id": tpl["window_id"], "timestamp": tpl["ts"],
                            "link_id": tpl["link_id"].astype(str), "queue_pred": qp})
        parts.append(res)
        log(f"  T2 predict [{P.name}/{split}] rows {len(res):,}, positive rate {qp.mean():.4f}")
    return pd.concat(parts, ignore_index=True)

## TASK 4 (baseline passthrough) + inspection helper

In [ ]:
def run_task4_baseline():
    script = REPO / "src/task4/build_task4_odme_artifacts.py"
    parts = []
    for split in ("validation", "private"):
        out = OUT / f"odme_{split}"
        out.mkdir(parents=True, exist_ok=True)
        if not (out / "baseline_submission.csv").exists():
            subprocess.run([sys.executable, str(script), "--release-root", str(RELEASE),
                            "--split", split, "--output-root", str(out)], check=True, cwd=str(REPO))
        df = pd.read_csv(out / "baseline_submission.csv"); df["split"] = split
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


def inspect_task4():
    lines = []
    root = RELEASE / "task4"
    for p in sorted(root.rglob("*"))[:200]:
        if p.is_file():
            lines.append(f"{p.relative_to(RELEASE)}  ({p.stat().st_size:,} bytes)")
    seen = set()
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix in (".csv", ".parquet") and p.name not in seen:
            seen.add(p.name)
            try:
                df = pd.read_csv(p, nrows=4) if p.suffix == ".csv" else pd.read_parquet(p).head(4)
                lines.append(f"\n=== {p.relative_to(RELEASE)} cols={df.columns.tolist()}\n{df.to_string()}")
            except Exception as e:
                lines.append(f"\n=== {p}: {e}")
    for rd in (REPO / "src" / "task4").glob("*.py"):
        lines.append(f"\n### {rd.name}: " + ", ".join(re.findall(r"^def (\w+)", rd.read_text(), re.M)))
    txt = "\n".join(lines)
    (OUT / "task4_inspect.txt").write_text(txt)
    print(txt)

## merge into the single upload file

In [ ]:
def build_submission(state, queue, odme):
    key_cols = ["submission_id", "task", "split", "panel", "timestamp", "station_id", "link_id",
                "mask_regime", "window_id", "departure_time", "path_id", "origin_zone", "destination_zone"]
    key = pd.read_csv(RELEASE / "submission_key.csv", usecols=key_cols,
                      dtype={c: "string" for c in key_cols if c != "submission_id"})
    sub = pd.read_csv(RELEASE / "sample_submission.csv")
    assert set(key["submission_id"]) == set(sub["submission_id"])
    pos = pd.Series(np.arange(len(sub)), index=sub["submission_id"].to_numpy())
    key["timestamp"] = pd.to_datetime(key["timestamp"], utc=True, errors="coerce")

    def attach(task, pred, join, cols):
        kt = key[key["task"] == task]
        if pred.duplicated(join).any():
            raise ValueError(f"duplicate keys in {task} predictions")
        m = kt.merge(pred[join + cols], on=join, how="left", validate="one_to_one")
        bad = m[cols].isna().any(axis=1).sum()
        if bad:
            raise ValueError(f"{task}: {bad:,} rows without prediction")
        idx = pos.loc[m["submission_id"].to_numpy()].to_numpy()
        for c in cols:
            arr = sub[c].to_numpy(copy=True).astype(float)
            arr[idx] = m[c].to_numpy(float)
            sub[c] = arr
        log(f"  merged {task}: {len(m):,}")

    attach("state", state, ["split", "panel", "timestamp", "station_id", "link_id", "mask_regime"],
           ["speed_kmh", "flow_vph"])
    attach("queue", queue, ["split", "window_id", "timestamp", "link_id"], ["queue_pred"])
    o = odme.copy()
    for c in ("panel", "departure_time", "path_id", "origin_zone", "destination_zone"):
        o[c] = o[c].astype(str)
        key[c] = key[c].astype("string").astype(str)
    attach("odme", o, ["split", "panel", "departure_time", "path_id", "origin_zone", "destination_zone"],
           ["path_flow"])
    sub["queue_pred"] = sub["queue_pred"].round().clip(0, 1).astype(int)
    for c in ("speed_kmh", "flow_vph", "path_flow"):
        sub[c] = np.clip(np.nan_to_num(sub[c].to_numpy(float), nan=0.0, posinf=0.0), 0, None)
    return sub


# ----------------------------------------------------------------------------
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true")
    ap.add_argument("--inspect4", action="store_true")
    ap.add_argument("--panels", default="")
    ap.add_argument("--resume", action="store_true", help="reuse cached predictions in OUT")
    a = ap.parse_args(argv)
    OUT.mkdir(parents=True, exist_ok=True)
    ensure_repo()
    if a.inspect4:
        inspect_task4(); return
    names = sorted(p.name for p in (RELEASE / "task1").iterdir() if p.is_dir())
    if a.panels:
        names = [n for n in names if n in a.panels.split(",")]
    if a.quick:
        # D12_I405_* are excluded from Task 2, so pick two other panels
        names = [n for n in names if not n.startswith("D12_I405")][:2]
    log("panels:", names)
    panels = [Panel(n, i) for i, n in enumerate(names)]

    pre = "quick_" if a.quick else ""
    sp_path, qp_path = OUT / f"{pre}state_pred.parquet", OUT / f"{pre}queue_pred.parquet"
    if a.resume and sp_path.exists():
        state = pd.read_parquet(sp_path)
    else:
        log("== TASK 1 =="); models, _ = train_task1(panels, a.quick)
        state = pd.concat([predict_task1(panels, models, s) for s in ("validation", "private")], ignore_index=True)
        state.to_parquet(sp_path)
    if a.resume and qp_path.exists():
        queue = pd.read_parquet(qp_path)
    else:
        log("== TASK 2 =="); model, rule = train_task2(panels, a.quick)
        queue = pd.concat([predict_task2(panels, model, rule, s) for s in ("validation", "private")], ignore_index=True)
        queue.to_parquet(qp_path)
    if a.quick:
        log("quick mode finished (no final merge)."); return
    log("== TASK 4 (baseline) =="); odme = run_task4_baseline()
    log("== MERGE ==")
    sub = build_submission(state, queue, odme)
    sub.to_csv(WORK / "submission.csv", index=False)
    log("saved", WORK / "submission.csv", sub.shape)

## Quick test (2 non-I405 panels, no final merge)

In [ ]:
main(["--quick"])

## Full run → `/kaggle/working/submission.csv`
(`--resume` reuses cached predictions in `tfb_out/` if a previous full run stopped midway.)

In [ ]:
main([])  # or main(["--resume"])

## Task 4 file inspection (send this output to refine Task 4)

In [ ]:
ensure_repo()
inspect_task4()